In [ ]:
# Exploratory Data Analysis

### Sections:
# 1. Setup & Dataset Overview
# 2. Risk Level Distribution
# 3. Fire Cause Analysis
# 4. Building Characteristics vs Risk Level
# 5. Geographic Patterns
# 6. Correlation Heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# Load merged dataset
df = pd.read_csv('../data/processed/merged_calfire.csv')

# Map numeric CAL FIRE cause codes to readable labels
cause_map = {
    1.0:  'Lightning',
    2.0:  'Equipment Use',
    3.0:  'Smoking',
    4.0:  'Campfire',
    5.0:  'Debris Burning',
    7.0:  'Arson',
    8.0:  'Playing with Fire',
    9.0:  'Electrical Power',
    10.0: 'Vehicle',
    11.0: 'Miscellaneous',
    14.0: 'Undetermined',
    15.0: 'Structure Fire',
}
df['cause'] = df['cause'].map(cause_map).fillna('Other')

print(f" Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
display(df.head(3))

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total buildings inspected : {len(df):,}")
print(f"Total columns             : {df.shape[1]}")
print(f"Years covered             : {int(df['fire_year'].min())} – {int(df['fire_year'].max())}")
print(f"Unique fires              : {df['fire_name'].nunique():,}")
print(f"Unique counties           : {df['county'].nunique()}")
print(f"\nColumn data types:")
print(df.dtypes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Order for all charts
risk_order = ['High', 'Medium', 'Low']
colors     = ['#d62728', '#ffdd57', '#2ca02c']

# Chart 1 — Count bar chart
counts = df['risk_level'].value_counts().reindex(risk_order)
axes[0].bar(risk_order, counts.values, color=colors)
axes[0].set_title('Risk Level Distribution (Count)')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Buildings')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Chart 2 — Percentage pie chart
axes[1].pie(
    counts.values,
    labels=risk_order,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140
)
axes[1].set_title('Risk Level Distribution (%)')

plt.suptitle('Target Variable — Risk Level', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n  Class imbalance noted:")
print(f"High = {counts['High']/len(df)*100:.1f}% → will fix with SMOTE")

In [ ]:
# Fire Cause Analysis

# --- Top Fire Causes (absolute count, coloured by risk mix) ---
plt.close('all')

top_causes = df['cause'].value_counts().head(10).index
df_cause = df[df['cause'].isin(top_causes)].copy()

cause_risk_ct = (
    df_cause.groupby(['cause', 'risk_level']).size()
    .unstack(fill_value=0)
)
for col in risk_order:
    if col not in cause_risk_ct.columns:
        cause_risk_ct[col] = 0
cause_risk_ct = cause_risk_ct[risk_order]

# Sort causes by total buildings (descending)
cause_risk_ct = cause_risk_ct.loc[cause_risk_ct.sum(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(12, 6))
cause_risk_ct.plot(kind='barh', stacked=True, color=colors, ax=ax, width=0.72)
ax.invert_yaxis()
ax.set_title('Top 10 Fire Causes — Buildings Affected by Risk Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Buildings')
ax.set_ylabel('Fire Cause')
ax.legend(title='Risk Level', bbox_to_anchor=(1.01, 1), loc='upper left')

# Add count labels on total bar end
totals = cause_risk_ct.sum(axis=1)
for i, (idx, total) in enumerate(totals.items()):
    ax.text(total + 200, i, f'{total:,}', va='center', fontsize=9, color='#333')

plt.tight_layout()
plt.savefig('../outputs/03a_cause_count.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
#  Building Characteristics vs Risk Level

# --- Clean before plotting ---

# 1. Drop nulls and empty/whitespace strings
df_plot = df.copy()
df_plot['roof_construction'] = df_plot['roof_construction'].astype(str).str.strip()
df_plot['structure_type']    = df_plot['structure_type'].astype(str).str.strip()

# 2. Replace known junk values with NaN, then drop
junk = {'nan', 'none', 'null', '0', '', 'unknown', 'not applicable'}
df_plot.loc[df_plot['roof_construction'].str.lower().isin(junk), 'roof_construction'] = None
df_plot.loc[df_plot['structure_type'].str.lower().isin(junk), 'structure_type'] = None

df_plot = df_plot.dropna(subset=['roof_construction', 'structure_type'])

# 3. Check what's actually left — run this first to see your real values
print("Roof types:\n", df_plot['roof_construction'].value_counts())
print("\nStructure types:\n", df_plot['structure_type'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Chart 1 — Top 8 roof types (drop Concrete Slab, No Deck/Porch, Non Combustible — too rare)
top_roofs = ['Asphalt', 'Metal', 'Fire Resistant', 'Tile', 'Other', 'Wood', 'Combustible', 'Concrete']

roof_risk = df_plot[df_plot['roof_construction'].isin(top_roofs)]\
              .groupby(['roof_construction', 'risk_level']).size()\
              .unstack(fill_value=0)
roof_risk_pct = roof_risk.div(roof_risk.sum(axis=1), axis=0) * 100
roof_risk_pct = roof_risk_pct.loc[
    sorted(top_roofs, key=lambda x: df_plot['roof_construction'].value_counts()[x], reverse=True)
]
roof_risk_pct[risk_order].plot(kind='bar', stacked=True, color=colors, ax=axes[0])
axes[0].set_title('Roof Type vs Risk Level')
axes[0].set_xlabel('')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right', fontsize=9)
axes[0].legend(title='Risk Level')

# Chart 2 — Keep only structure types with ≥500 buildings (drops Hospital, Agriculture, etc.)
top_structs = [
    'Single Family Residence Single Story',
    'Utility Misc Structure',
    'Single Family Residence Multi Story',
    'Mobile Home Double Wide',
    'Mobile Home Single Wide',
    'Commercial Building Single Story',
    'Motor Home',
    'Multi Family Residence Multi Story',
    'Multi Family Residence Single Story',
    'Mobile Home Triple Wide',
    'Infrastructure',
    'Commercial Building Multi Story',
    'School',
    'Mixed Commercial/Residential',
]

# Shorten long labels for readability
label_map = {
    'Single Family Residence Single Story':  'SFR Single Story',
    'Single Family Residence Multi Story':   'SFR Multi Story',
    'Utility Misc Structure':                'Utility/Misc',
    'Mobile Home Double Wide':               'Mobile Home DW',
    'Mobile Home Single Wide':               'Mobile Home SW',
    'Mobile Home Triple Wide':               'Mobile Home TW',
    'Commercial Building Single Story':      'Commercial SS',
    'Commercial Building Multi Story':       'Commercial MS',
    'Multi Family Residence Multi Story':    'MFR Multi Story',
    'Multi Family Residence Single Story':   'MFR Single Story',
    'Mixed Commercial/Residential':          'Mixed Comm/Res',
    'Infrastructure':                        'Infrastructure',
    'School':                                'School',
    'Motor Home':                            'Motor Home',
}

struct_risk = df_plot[df_plot['structure_type'].isin(top_structs)]\
                .groupby(['structure_type', 'risk_level']).size()\
                .unstack(fill_value=0)
struct_risk_pct = struct_risk.div(struct_risk.sum(axis=1), axis=0) * 100
struct_risk_pct = struct_risk_pct.loc[
    sorted(top_structs, key=lambda x: df_plot['structure_type'].value_counts()[x], reverse=True)
]
struct_risk_pct.index = struct_risk_pct.index.map(label_map)
struct_risk_pct[risk_order].plot(kind='bar', stacked=True, color=colors, ax=axes[1])
axes[1].set_title('Structure Type vs Risk Level')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].legend(title='Risk Level')

plt.suptitle('Building Characteristics vs Risk Level', fontsize=14, fontweight='bold')
plt.subplots_adjust(bottom=0.35, wspace=0.3)
plt.savefig('../outputs/04_building_characteristics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#  County vs Risk Level

plt.close('all')

top_counties = df['county'].value_counts().head(15).index
df_county = df[df['county'].isin(top_counties)]

county_risk = df_county.groupby(['county', 'risk_level']).size()\
                       .unstack(fill_value=0)
county_risk_pct = county_risk.div(county_risk.sum(axis=1), axis=0) * 100

# Sort by % High risk descending so most dangerous counties are at top
county_risk_pct = county_risk_pct.loc[
    county_risk_pct['High'].sort_values(ascending=True).index
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Chart 1 — % stacked bar by risk level
county_risk_pct[risk_order].plot(
    kind='barh', stacked=True, color=colors, ax=axes[0], width=0.75
)
axes[0].set_title('Risk Level Distribution by County\n(Top 15 counties by building count)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Percentage (%)')
axes[0].set_ylabel('')
axes[0].legend(title='Risk Level', bbox_to_anchor=(1.01, 1), loc='upper left')

# Add % High label at end of each bar
for i, (county, row) in enumerate(county_risk_pct.iterrows()):
    axes[0].text(101, i, f"{row['High']:.0f}%",
                 va='center', fontsize=8, color='#d62728', fontweight='bold')

# Chart 2 — Total building count per county (context for chart 1)
county_totals = df_county['county'].value_counts().loc[county_risk_pct.index]
axes[1].barh(range(len(county_totals)), county_totals.values,
             color='#4878cf', alpha=0.75)
axes[1].set_yticks(range(len(county_totals)))
axes[1].set_yticklabels(county_totals.index, fontsize=9)
axes[1].set_title('Total Buildings Inspected\nby County',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Number of Buildings')
axes[1].set_ylabel('')

for i, v in enumerate(county_totals.values):
    axes[1].text(v + 200, i, f'{v:,}', va='center', fontsize=8)

plt.suptitle('Geographic Risk Distribution — Top 15 Counties',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/08_county_risk.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 9. Correlation Heatmap

plt.close('all')

# Encode risk_level as numeric so it appears in heatmap as the target
df['risk_numeric'] = df['risk_level'].map({'High': 2, 'Medium': 1, 'Low': 0})

numeric_cols = [
    'risk_numeric',           # target variable
    'assessed_value',         # building-level
    'year_built',             # building-level
    'gis_acres',              # fire-level*
    'area_burned',            # fire-level*
    'fire_duration_days',     # fire-level*
    'homes_destroyed',        # fire-level*
    'financial_loss_million', # fire-level*
    'fatalities',             # fire-level*
]

# * fire-level columns are repeated per building in same fire
#   so correlations between them will be artificially inflated

corr = df[numeric_cols].corr()

# Rename for cleaner labels
label_map = {
    'risk_numeric':           'Risk Level',
    'assessed_value':         'Assessed Value',
    'year_built':             'Year Built',
    'gis_acres':              'GIS Acres',
    'area_burned':            'Area Burned',
    'fire_duration_days':     'Fire Duration',
    'homes_destroyed':        'Homes Destroyed',
    'financial_loss_million': 'Financial Loss ($M)',
    'fatalities':             'Fatalities',
}
corr = corr.rename(index=label_map, columns=label_map)

fig, ax = plt.subplots(figsize=(11, 9))

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    annot_kws={'size': 9}
)

ax.set_title('Correlation Heatmap — Numeric Features vs Risk Level',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

# Note about fire-level columns
fig.text(0.5, -0.02,
         '* Area Burned, Fire Duration, Homes Destroyed, Financial Loss, Fatalities are fire-level values\n'
         '  repeated per building — correlations between these columns are inflated.',
         ha='center', fontsize=8, color='gray', style='italic')

plt.tight_layout()
plt.savefig('../outputs/09_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print top correlations with Risk Level specifically
print("\nTop correlations with Risk Level:")
risk_corr = corr['Risk Level'].drop('Risk Level').sort_values(key=abs, ascending=False)
for feat, val in risk_corr.items():
    bar = '█' * int(abs(val) * 20)
    print(f"  {feat:<22} {val:+.3f}  {bar}")